In [1]:
import spikeinterface as si
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from kilosort import io, run_kilosort
import pandas as pd
import neo.rawio
import kilosort
from kilosort.data_tools import get_best_channels
import gc
import torch


In [2]:
gc.collect()

16334

In [3]:
torch.cuda.empty_cache()

In [4]:
def makeProbeFromFile(chanMapDict, spacing=(1,1), grouped=0, chnOffset =0):
    chanMap = pd.read_excel(chanMapDict['filename'],sheet_name = chanMapDict['sheet']).iloc[chanMapDict['cellIdx'][0], chanMapDict['cellIdx'][1]]
    chanMap = np.rot90(np.flipud(chanMap),-1)
    [xc, yc] = [np.where(pd.notna(chanMap))[1], np.where(pd.notna(chanMap))[0]]
    
    xc = xc*spacing[0]
    yc = yc*spacing[1]
    
    n_chan = len(xc)
    connected = np.ones((n_chan,1)).astype('bool')
    
    chanMap = np.arange(n_chan) + chnOffset
    
    if grouped:
        kcoords = np.ones(n_chan)
    else:
        kcoords = np.arange(n_chan)

    probe = {
        'chanMap': chanMap,
        'xc': xc,
        'yc': yc,
        'kcoords': kcoords,
        'n_chan': n_chan
    }
    return probe

def makeProbeFromParameters(n_chan, spacing, grouped=0, chnOffset= 0, x0=0, y0=0):
    chanMap = np.arange(n_chan) + chnOffset
    xc = np.array([x0 + spacing[0]*i for i in reversed(range(n_chan))])
    yc = np.array([y0 + spacing[1]*i for i in reversed(range(n_chan))]) # reversed makes things identical to how they were in matlab
    
    if grouped:
        kcoords = np.ones(n_chan)
    else:
        kcoords = np.arange(n_chan)
    
    
    probe = {
        'chanMap': chanMap,
        'xc': xc,
        'yc': yc,
        'kcoords': kcoords,
        'n_chan': n_chan
    }
    return probe
    


In [5]:
def getProbeSubset(probe, channels):
    keys = ['chanMap', 'xc', 'yc', 'kcoords']
    probeSubset = {k: probe[k][channels] for k in keys}
    probeSubset['n_chan'] = len(channels)
    return probeSubset

In [6]:
#monkey_name = "Sprout"
#exptDate = "260205"
#pl2path = Path.home() / "Data" / "V1_Fovea" / monkey_name / exptDate
dataPath = Path.home() / "Data" / "V1_Fovea"


processingPath = Path.home() / "Git" / "ConwayExptProcessing"
#configPath = processingPath / "chanMapFiles" / monkey_name
# find plexon filename

#plexon_fname =  pl2path / (exptDate + "_153540_" + monkey_name + ".pl2")


plexon_fname = dataPath / "Jocamo" / "260403" / "260403_141736_Jacomo.pl2"
assert plexon_fname.is_file()

#fname = "/home/bizon/Data/V1_Fovea/Sprout/260205/260205_153540_Sprout.pl2"


96

In [8]:
#chanMapPath = processingPath / "chanMapFiles" / monkey_name


chanMapDicts = [{'filename': chanMapPath / 'SN 7623-000080 Mapping and Impedance' / 'SN 7623-000080.xlsm',
                'sheet': 'Z check',
                'cellIdx': [np.arange(14, 25), np.arange(15,25)]},
               {'filename': chanMapPath / 'SN 7624-000191 Mapping and Impedance' /'1125-15 SN 7624-000191.xlsm',
               'sheet': 'Z check',
               'cellIdx':[np.arange(14, 25), np.arange(15,25)]}, {}]


NameError: name 'monkey_name' is not defined

In [13]:
arrayDict['chanMapDict'][0]

IndexError: list index out of range

In [7]:

# load probe info or make it if it doesnt exist


# arrayDict = {'labels': ['UT1', 'UT2', 'lam'],
#             'n_chan': [(), (), 64],
#             'spacing': [(400,400), (400,400), (0,50)],
#              'grouped': [1,1,1],
#             'chanMapDict': chanMapDicts}


labels = ['lam']
n_chan = [64]
spacing = [(0,50)]
grouped = [1]
chanMapDict = []


arrayDict = {'labels': labels,
            'n_chan': n_chan,
            'spacing': spacing,
            'grouped': grouped,
            'chanMapDict': []}

# for each channel, determine if it has a channel map and if not specifiy geometry
probeDicts = []
chnOffset = 0

for i in range(len(arrayDict['labels'])):
    # if this array has an excel file, make probe info from file
    if len(arrayDict['chanMapDict']) > 0:
        chanMapDict = arrayDict['chanMapDict'][i]
    else:
        chanMapDict = []
    
    if chanMapDict:
        probeDicts.append(makeProbeFromFile(chanMapDict=chanMapDict,
                                            spacing=arrayDict['spacing'][i],
                                            grouped=arrayDict['grouped'][i],
                                            chnOffset=chnOffset)) 
        chnOffset = chnOffset + probeDicts[i]['n_chan']
    else:
        probeDicts.append(makeProbeFromParameters(n_chan=arrayDict['n_chan'][i],
                                                  spacing=arrayDict['spacing'][i],
                                                  grouped=arrayDict['grouped'][i],
                                                  chnOffset=chnOffset))
        chnOffset = chnOffset + probeDicts[i]['n_chan']
            

In [8]:
probeDicts

[{'chanMap': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
         17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
         34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
         51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
  'xc': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  'yc': array([3150, 3100, 3050, 3000, 2950, 2900, 2850, 2800, 2750, 2700, 2650,
         2600, 2550, 2500, 2450, 2400, 2350, 2300, 2250, 2200, 2150, 2100,
         2050, 2000, 1950, 1900, 1850, 1800, 1750, 1700, 1650, 1600, 1550,
         1500, 1450, 1400, 1350, 1300, 1250, 1200, 1150, 1100, 1050, 1000,
          950,  900,  850,  800,  750,  700,  650,  600,  550,  500,  450,
          400,  350,  300,  250,  200,  150,  100,   50,    0]),
  'kcoords': array([1.

In [15]:
SAVE_PATH = dataPath / "Jocamo" /"260403" / "260403_141736_Jacomo" / "kilosort_laminar_1to64" 

In [16]:
probeSubset = getProbeSubset(probeDicts[0], range(64))
probeSubset

{'chanMap': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
 'xc': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'yc': array([3150, 3100, 3050, 3000, 2950, 2900, 2850, 2800, 2750, 2700, 2650,
        2600, 2550, 2500, 2450, 2400, 2350, 2300, 2250, 2200, 2150, 2100,
        2050, 2000, 1950, 1900, 1850, 1800, 1750, 1700, 1650, 1600, 1550,
        1500, 1450, 1400, 1350, 1300, 1250, 1200, 1150, 1100, 1050, 1000,
         950,  900,  850,  800,  750,  700,  650,  600,  550,  500,  450,
         400,  350,  300,  250,  200,  150,  100,   50,    0]),
 'kcoords': array([1., 1., 1., 1., 

In [17]:
fs = 40000.0
highpass_cutoff = 300.0
Th_universal = 6.0
Th_learned = 9.0
#nt = 81
#n_pcs = 3

# for utah arrays
n_chan_bin = 64
dmin = 50 #400 for Utah
#dminx = 32 # 400 for Utah

nblocks = 0 # 0 for Utah
x_centers = None # 4for Utah

settings = {
    "filename": SAVE_PATH,
    "n_chan_bin": n_chan_bin,
    "fs": fs,
    "highpass_cutoff": highpass_cutoff,
    "batch_size": int(2*fs)   # 2.5 for 32 chan   
#     "nblocks": nblocks,           
#     "Th_universal": Th_universal,
#     "Th_learned": Th_learned,
#     "x_centers": x_centers,
#     "nearest_templates": 32,
#     "dmin": dmin
}

# settings["x_centers"] = np.linspace(
#     np.min(probeSubset["xc"]),
#     np.max(probeSubset["xc"]),
#     4  
# )

ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate, kept_spikes = run_kilosort(
    settings=settings,
    probe=probeSubset,
    filename= SAVE_PATH / '260403_141736_Jacomo_laminar_1to64.dat',
    results_dir = SAVE_PATH,
    data_dtype="int16",
    do_CAR=True,
    invert_sign=False,
    save_preprocessed_copy=True,
    clear_cache=True
)


kilosort.run_kilosort: Kilosort version 4.1.7
kilosort.run_kilosort: Python version 3.9.23
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: System information:
kilosort.run_kilosort: Linux-6.5.0-41-generic-x86_64-with-glibc2.35 x86_64
kilosort.run_kilosort: x86_64
kilosort.run_kilosort: Using GPU for PyTorch computations. Specify `device` to change this.
kilosort.run_kilosort: Using CUDA device: NVIDIA GeForce RTX 4080 15.68GB
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: Sorting [PosixPath('/home/bizon/Data/V1_Fovea/Jocamo/260403/260403_141736_Jacomo/kilosort_laminar_1to64/260403_141736_Jacomo_laminar_1to64.dat')]
kilosort.run_kilosort: clear_cache=True
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage before sorting
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     2.20 %
kilosort.run_kilosort: Mem used:     48.30 %     |     

kilosort.run_kilosort: GPU memory:   12.62 %     |      1.98   /    15.68 GB
kilosort.run_kilosort: Allocated:     0.30 %     |      0.05   /    15.68 GB
kilosort.run_kilosort: Max alloc:     3.08 %     |      0.48   /    15.68 GB
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort:  
kilosort.run_kilosort: First clustering
kilosort.run_kilosort: ----------------------------------------
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [03:34<00:00, 214.02s/it]
kilosort.run_kilosort: 146 clusters found, in 214.44s; total 1752.67s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after first clustering
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     8.30 %

,cluster_id,ch,n_spikes,KSLabel_x,KSLabel_y
0,0,9,159518,mua,mua
1,1,10,110688,mua,mua
2,2,9,32946,mua,mua
3,3,9,178416,mua,mua
4,4,19,100793,mua,mua
5,5,29,674664,mua,mua
6,6,30,98248,mua,mua
7,7,29,139071,mua,mua
8,8,19,198300,mua,mua
9,9,3,162923,mua,mua


In [9]:
raw = np.fromfile(SAVE_PATH / "UT1.dat", dtype=np.int16, count=96*2000)

# Extract "channel 0" assuming Kilosort format
ch0 = raw[0::96]

# Extract "channel 0" assuming YOUR format
ch0_alt = raw[:2000]

print(ch0[:20])
print(ch0_alt[:20])

[-17 -43 -56 -16 -35 -28 -22 -13  15   5   4   6   7  23  24  34  51 -42
  -8  36]
[-17  -3  15  14  21   7  21  21  -3 -17  22 -48   1 -19  18  35  13 -28
  44  42]


In [34]:
print(probeDicts[0])

{'chanMap': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]), 'xc': array([ 400,  800, 1200, 1600, 2000, 2400, 2800, 3200, 3600,    0,  400,
        800, 1200, 1600, 2000, 2400, 2800, 3200, 3600,    0,  400,  800,
       1200, 1600, 2000, 2400, 2800, 3200, 3600,    0,  400,  800, 1200,
       1600, 2000, 2400, 2800, 3200, 3600,    0,  400,  800, 1200, 1600,
       2000, 2400, 2800, 3200,    0,  400,  800, 1200, 1600, 2000, 2400,
       2800, 3200, 3600,    0,  400,  800, 1200, 1600, 2000, 2400, 2800,
       3200, 3600,    0,  400,  800, 1200, 1600, 2000, 2400, 2800, 3200,
       3600,    0,  400,  800, 1200, 1600, 

In [15]:
print(np.fromfile(SAVE_PATH/"UT1.dat", dtype=np.int16).shape)

data = np.fromfile(SAVE_PATH/"UT1.dat", 
                   dtype=np.int16,
                   count=100,
                offset=int(1e9))

print(data)

(11874168960,)
[  5 -12  29   6  -2  17   3 -36  21 -10 -19  18  16 -12   1  28 -33  -3
   5 -47  23  20  -7 -38 -54  13   0  16 -11 -38   0  12  24  21  16   8
  -5  10 -17 -37   4  -8 -16  36  19 -19   8  42 -54   6   2 -47  35   9
 -20 -30 -26   3 -28  25  -8 -21   5  17  26  21   0  20   1  13 -30 -32
 -14 -11 -13  41  23 -10  21  11 -48  13  -3 -37  33   5 -30 -26  14  -3
 -51  38   0   7   8  18  35  -1  -8  23]


In [9]:
print(np.fromfile(pl2path/'260205_153540_Sprout'/'kilosorting_UT1_1to32'/'260205_153540_Sprout.dat', dtype=np.int16).shape)

data = np.fromfile(pl2path/'260205_153540_Sprout'/'kilosorting_UT1_1to32'/'260205_153540_Sprout.dat', 
                   dtype=np.int16,
                   count=100,
                offset=int(1e9))

print(data)

(11874168960,)
[  5 -12  29   6  -2  17   3 -36  21 -10 -19  18  16 -12   1  28 -33  -3
   5 -47  23  20  -7 -38 -54  13   0  16 -11 -38   0  12  24  21  16   8
  -5  10 -17 -37   4  -8 -16  36  19 -19   8  42 -54   6   2 -47  35   9
 -20 -30 -26   3 -28  25  -8 -21   5  17  26  21   0  20   1  13 -30 -32
 -14 -11 -13  41  23 -10  21  11 -48  13  -3 -37  33   5 -30 -26  14  -3
 -51  38   0   7   8  18  35  -1  -8  23]


In [ ]:
#recording = se.read_plexon2(plexon_fname, stream_id = 'SPKC')

In [ ]:
#recording = se.read_plexon2(fname, stream_id='AI')
#recording_ai01 = si.ChannelSliceRecording(recording, channel_ids=['AI01'])
#traces = recording_ai01.get_traces()
#x = np.linspace(0, (len(traces)-1)/1000, len(traces))
#plt.plot(x[:10000], traces[:10000])
#plt.show()

### 